# 09 – Actuator Saturation

**Manufacturing context:** Every motor has a maximum torque. When the controller requests more force than the actuator can deliver, the control signal clips — this is **saturation**. It changes the closed-loop behavior and can cause unexpected overshoot.

This notebook compares PID response with and without actuator limits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Editable parameters ----
m = 1.0
c = 1.0
k = 4.0
ref = 1.0
t_end = 10.0
dt = 0.001

Kp = 30.0
Ki = 15.0
Kd = 8.0

# Actuator limits
u_min = -50.0
u_max = 50.0
# -----------------------------

In [ ]:
t = np.arange(0, t_end, dt)

def simulate_pid_sat(Kp, Ki, Kd, u_min=None, u_max=None):
    x = np.zeros_like(t)
    v = np.zeros_like(t)
    u = np.zeros_like(t)
    integral_e = 0.0
    prev_e = 0.0

    for i in range(1, len(t)):
        e = ref - x[i-1]
        integral_e += e * dt
        derivative_e = (e - prev_e) / dt

        u_raw = Kp * e + Ki * integral_e + Kd * derivative_e

        # Clip to actuator limits
        if u_min is not None and u_max is not None:
            u[i] = np.clip(u_raw, u_min, u_max)
        else:
            u[i] = u_raw

        prev_e = e

        a = (u[i] - c * v[i-1] - k * x[i-1]) / m
        v[i] = v[i-1] + a * dt
        x[i] = x[i-1] + v[i] * dt

    return x, u

x_unsat, u_unsat = simulate_pid_sat(Kp, Ki, Kd)
x_sat, u_sat = simulate_pid_sat(Kp, Ki, Kd, u_min, u_max)

print(f"Actuator limits: [{u_min}, {u_max}] N")
print(f"Peak control (no limits) : {np.max(np.abs(u_unsat)):.1f} N")
print(f"Peak control (saturated) : {np.max(np.abs(u_sat)):.1f} N")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

axes[0].plot(t, x_unsat, label="No limits")
axes[0].plot(t, x_sat, label="Saturated")
axes[0].axhline(ref, color="k", linestyle="--", linewidth=0.8, label="Reference")
axes[0].set_ylabel("Position [m]")
axes[0].set_title("Effect of Actuator Saturation on PID Response")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, u_unsat, label="No limits")
axes[1].plot(t, u_sat, label="Saturated")
axes[1].axhline(u_max, color="r", linestyle=":", linewidth=0.8, alpha=0.5)
axes[1].axhline(u_min, color="r", linestyle=":", linewidth=0.8, alpha=0.5)
axes[1].set_ylabel("Control effort [N]")
axes[1].set_xlabel("Time [s]")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### Student Exercise

1. Lower `u_max` to 20 N. How much slower is the response?
2. Look at the saturated control effort plot. Why does the integrator cause problems here? (Hint: next notebook.)
3. On a real servo drive, where would you find the torque limit setting?